In [ ]:
import os
import numpy as np
import pandas as pd

from gudhi.representations import BettiCurve
from scipy.ndimage import gaussian_filter1d


base = "RIPS"
RESOLUTION = 150
SIGMA = 2
NORMALIZATION = "l1"
THR_OPT = "p10"
DIMS = ["01"]
EPS = 1e-12



def read_and_save(filedir, tube):
    if tube and tube[0] != ".":
        _, ext = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split("_")[-1].split(".")[0]
        if ext != ".pdf" and tubenamerips == "Rips0":
            r0_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips0.txt")
            r1_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips1.txt")

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []


def list_patients(root_dir):
    return [x for x in sorted(os.listdir(root_dir)) if not x.startswith(".")]


def find_rips0_file(patient_dir):
    for f in sorted(os.listdir(patient_dir)):
        if f.startswith("."):
            continue
        if f.endswith("_Rips0.txt"):
            return f
    return None


def load_group_diagrams(group_root):
    out = {}
    for patient in list_patients(group_root):
        p_dir = os.path.join(group_root, patient)
        rips0_file = find_rips0_file(p_dir)
        if rips0_file is None:
            continue
        data = read_and_save(p_dir, rips0_file)
        if not data:
            continue
        diagrams = data[0]
        if diagrams is None or len(diagrams) < 2:
            continue
        out[patient] = diagrams
    return out


def collect_persistences(diagrams_list, dim):
    pers = []
    for diagrams in diagrams_list:
        pairs = diagrams[dim]
        if pairs is None or len(pairs) == 0:
            continue
        pairs = np.asarray(pairs, dtype=float)
        p = pairs[:, 1] - pairs[:, 0]
        p = p[np.isfinite(p)]
        p = p[p > 0]
        if p.size:
            pers.append(p)
    return np.concatenate(pers) if pers else np.array([], dtype=float)


def apply_persistence_threshold(pairs, thr):
    if pairs is None or len(pairs) == 0:
        return np.zeros((0, 2), float)
    pairs = np.asarray(pairs, float)
    pers = pairs[:, 1] - pairs[:, 0]
    keep = np.isfinite(pers) & (pers >= thr)
    return pairs[keep]


def betti_curve_from_pairs(pairs, resolution):
    if len(pairs) == 0:
        return np.zeros(resolution)
    bc = BettiCurve(resolution=resolution)
    return bc.fit_transform([pairs])[0].astype(float)


def normalize_curve(c, mode):
    if mode == "none":
        return c
    if mode == "l1":
        s = np.sum(np.abs(c))
        return c / (s + EPS)
    return c


def smooth_curve(c, sigma):
    if sigma <= 0:
        return c
    return gaussian_filter1d(c, sigma=float(sigma), mode="nearest")



if __name__ == "__main__":

    NR_root = os.path.join(base, "NonRelapse")
    R_root  = os.path.join(base, "Relapse")

    NR_diagrams = load_group_diagrams(NR_root)
    R_diagrams  = load_group_diagrams(R_root)

    listdirNR = sorted(NR_diagrams.keys())
    listdirR  = sorted(R_diagrams.keys())

    X = [NR_diagrams[p] for p in listdirNR] + [R_diagrams[p] for p in listdirR]
    y = np.array([0]*len(listdirNR) + [1]*len(listdirR))
    patients = listdirNR + listdirR

    # thresholds p10
    thr = {}
    for d in [0,1]:
        pers = collect_persistences(X, d)
        thr[d] = float(np.percentile(pers,10)) if pers.size else 0.0


    BettiNR = []
    BettiR  = []

    for i, (pid, diagrams) in enumerate(zip(patients, X)):
        parts = []

        for d in [0,1]:
            pairs = diagrams[d]
            fpairs = apply_persistence_threshold(pairs, thr[d])

            c = betti_curve_from_pairs(fpairs, RESOLUTION)
            c = normalize_curve(c, NORMALIZATION)
            c = smooth_curve(c, SIGMA)

            parts.append(c)

        feat = np.concatenate(parts)

        if y[i] == 0:
            BettiNR.append(feat)
        else:
            BettiR.append(feat)

    BettiNR = np.asarray(BettiNR)
    BettiR  = np.asarray(BettiR)


    folder = f"BettiCurves01"
    subfolder = os.path.join(base, folder)

    os.makedirs(os.path.join(subfolder, "Relapse"), exist_ok=True)
    os.makedirs(os.path.join(subfolder, "NonRelapse"), exist_ok=True)

    for i, curve in enumerate(BettiR):
        np.savetxt(os.path.join(subfolder, "Relapse", f"{listdirR[i]}.csv"), curve)

    for i, curve in enumerate(BettiNR):
        np.savetxt(os.path.join(subfolder, "NonRelapse", f"{listdirNR[i]}.csv"), curve)
